## **Логика ноутбука**

1. Установка и импорты (все библиотеки, включая numpy, faiss, sentence-transformers и т.д.)

2.  Конфигурация (пути, константы)

3. Вспомогательные функции (Chunk, make_header, classify_heading, ...)

4. Парсер (определение функции parse_doc_01)

5. Загрузка/парсинг чанков с кэшированием

6. Анализ чанков (статистика, заголовки, проверка мусора)

7. Функция просмотра чанков

8. Создание и сохранение эмбеддингов + FAISS (с выбором модели e5-small/e5-base)

9. Продвинутые функции поиска (dense, BM25, RRF, фильтрация, CrossEncoder rerank)

10. Демонстрация работы поиска

11. Подключение LLM (Saiga) и RAG-пайплайн

In [2]:
# @title 1. Установка зависимостей
!pip install pymupdf pdfplumber sentence-transformers faiss-cpu rank-bm25 -q   # -q для тихого режима
!pip install sentence-transformers faiss-cpu rank-bm25

# @title 2. Импорты
import json
import pdfplumber
import fitz  # это PyMuPDF
import re
import gc
import os
import pickle

from typing import Tuple, List
from dataclasses import dataclass
from typing import List, Optional
from collections import Counter
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 63.3 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# @title 2. Конфигурация парсера для документа "01 Подготовка к работе"

# ----- Пути к файлам -----
DOC_01_PATH = '/content/drive/MyDrive/VipNet_infotex/01 ViPNet Coordinator HW 5. Подготовка к работе.pdf'
INDEX_DIR   = '/content/drive/MyDrive/VipNet_infotex/rag_index'
os.makedirs(INDEX_DIR, exist_ok=True)

# ----- Геометрия страницы (в пунктах) -----
HEADER_ZONE = 60.0      # верхний колонтитул
FOOTER_ZONE = 65.0      # нижний колонтитул

# ----- Пороги размера шрифта для определения типа текста -----
SZ_BODY_MIN = 9.5       # меньше этого — таблицы или колонтитулы
SZ_TITLE_H1 = 20.0      # крупные заголовки разделов (Segoe UI Light)
SZ_TITLE_H3 = 9.8       # подзаголовки (Segoe UI Semibold Bold)

# ----- Параметры разбиения на чанки -----
CHUNK_SIZE = 700
OVERLAP    = 100
MIN_TEXT   = 15          # минимальная длина текста для сохранения чанка

# ----- Названия шрифтов (по вхождению подстроки) -----
LIGHT_FONTS    = ('light',)                               # светлые
SEMIBOLD_FONTS = ('semibold', 'bold', 'heavy', 'black')  # полужирные
MONO_FONTS     = ('mono', 'courier', 'consol', 'code', 'fixed', 'lucida', 'cascadia')  # моноширинные

# ----- Регулярные выражения для отсеивания мусора -----
RE_NOISE = re.compile(
    r'^\s*\d{1,4}\s*$'            # номер страницы
    r'|^\s*[©®]\s'                # копирайт
    r'|^\s*$'                     # пустая строка
    r'|^[-–—]{3,}\s*$'            # разделитель
    r'|^ФРКЕ\.\d+'                 # технический номер
    r'|^Copyright\s*\(c\)'
    r'|^ViPNet\s+Coordinator\s+HW\s+5\s*[|]'  # колонтитул
    r'|^Версия продукта\s+\d'
    r'|\.{4,}',                    # оглавление
    re.IGNORECASE
)

# ----- Глаголы для определения CLI-команд -----
CLI_VERBS = (
    'show ', 'set ', 'no ', 'clear ', 'debug ', 'ping ',
    'traceroute ', 'interface ', 'ip ', 'route ', 'crypto ',
    'tunnel ', 'nat ', 'access-list ', 'firewall ', 'vipnet ',
    'iplir ', 'inet ', 'machine ', 'admin ', 'vpn ',
    'coordinator ', 'exit', 'enable', 'disable', 'reload',
)

# ----- Признаки продолжения CLI-блока -----
RE_CLI_CONT = re.compile(
    r'^\s{2,}|^[–\-]\s|^\[|^<|^\{|^\|'
    r'|^Пример|^Описание|^Синтаксис'
    r'|^Параметр|^Режим|^Права'
    r'|^Примечание|^Связанные|^где\s'
)

print("✅ Конфигурация загружена")

✅ Конфигурация загружена


In [4]:
# @title 3. Классы и вспомогательные функции

# -----------------------------------------------------------
#  Класс для хранения чанка
# -----------------------------------------------------------
@dataclass
class Chunk:
    """Структура одного фрагмента текста с метаданными."""
    chunk_id:   int
    text:       str      # чистый текст → BM25 + ответ пользователю
    text_ctx:   str      # текст с метаданными → эмбеддинг
    doc_name:   str
    page:       int
    section:    str      # H1
    subsection: str      # H2/H3
    cmd:        str      # имя CLI-команды
    chunk_type: str      # 'text' | 'cli' | 'table' | 'heading'

# -----------------------------------------------------------
#  Функции для работы с заголовками и шрифтами
# -----------------------------------------------------------
def make_header(doc_name: str, page: int, section: str, subsection: str, cmd: str = '') -> str:
    """Формирует строку с метаданными для добавления в text_ctx."""
    parts = [f'Документ: {doc_name}', f'Страница: {page}']
    if section:    parts.append(f'Раздел: {section}')
    if subsection: parts.append(f'Подраздел: {subsection}')
    if cmd:        parts.append(f'Команда: {cmd}')
    return ' | '.join(parts) + '\n'

def font_is_light(font: str) -> bool:
    """Проверяет, относится ли название шрифта к светлым."""
    return any(l in font.lower() for l in LIGHT_FONTS)

def font_is_semibold(font: str, bold: bool) -> bool:
    """Проверяет, является ли шрифт полужирным (semibold/bold)."""
    return bold and any(s in font.lower() for s in SEMIBOLD_FONTS)

def font_is_mono(font: str) -> bool:
    """Проверяет, является ли шрифт моноширинным."""
    return any(m in font.lower() for m in MONO_FONTS)

def is_numbered_step(text: str) -> bool:
    """Определяет, является ли строка нумерованным шагом инструкции."""
    return bool(re.match(r'^\d+(\.\d+)*[\s.]', text))

def classify_heading(full: str, fsize: float, font: str, bold: bool) -> Optional[str]:
    """
    Определяет уровень заголовка на основе размера, шрифта и жирности.
    Возвращает 'h1', 'h2', 'h3' или None.
    """
    if len(full) < 2 or full[0].islower():
        return None
    if RE_NOISE.search(full):
        return None

    # H1/H2 — крупный светлый шрифт (не требует bold)
    if fsize >= SZ_TITLE_H1 and font_is_light(font):
        return 'h1' if fsize >= 26 else 'h2'

    # H3 — жирный semibold, но не мелкий и не нумерованный шаг
    if fsize >= SZ_TITLE_H3 and font_is_semibold(font, bold):
        if is_numbered_step(full):
            return None
        if len(full) > 120:
            return None
        return 'h3'

    return None

# -----------------------------------------------------------
#  Функции для разбиения текста на чанки
# -----------------------------------------------------------
def split_text(text: str, size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> List[str]:
    """Разбивает длинный текст на перекрывающиеся фрагменты."""
    if len(text) <= size:
        return [text] if len(text) >= MIN_TEXT else []
    result, start = [], 0
    while start < len(text):
        end = start + size
        if end < len(text):
            for sep in ['.\n', '. ', '\n\n', '\n']:
                pos = text.rfind(sep, start + MIN_TEXT, end)
                if pos > start:
                    end = pos + len(sep)
                    break
        chunk = text[start:end].strip()
        if len(chunk) >= MIN_TEXT:
            result.append(chunk)
        start = end - overlap
        if start >= len(text):
            break
    return result

# -----------------------------------------------------------
#  Функции для работы с PDF (fitz и pdfplumber)
# -----------------------------------------------------------
def get_avg_font_size(doc: fitz.Document, n_pages: int = 5) -> float:
    """Оценивает средний размер основного шрифта по первым страницам."""
    page_h = doc[0].rect.height
    sizes = []
    for i in range(min(n_pages, len(doc))):
        page_dict = doc[i].get_text('dict', flags=fitz.TEXT_PRESERVE_LIGATURES)
        for block in page_dict['blocks']:
            if block['type'] != 0:
                continue
            y0, y1 = block['bbox'][1], block['bbox'][3]
            if y1 <= HEADER_ZONE or y0 >= page_h - FOOTER_ZONE:
                continue
            for line in block['lines']:
                for span in line['spans']:
                    sz = span['size']
                    if 9.5 <= sz <= 14 and span['text'].strip():
                        sizes.append(sz)
        del page_dict
        gc.collect()
    return (sum(sizes) / len(sizes)) if sizes else 10.0

def get_tables_on_page(pdf_path: str, page_idx: int) -> List[str]:
    """Извлекает таблицы с указанной страницы с помощью pdfplumber."""
    tables = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            if page_idx >= len(pdf.pages):
                return tables
            page = pdf.pages[page_idx]
            cropped = page.crop((0, HEADER_ZONE, page.width, page.height - FOOTER_ZONE))
            for table in cropped.extract_tables() or []:
                rows = [
                    ' | '.join(str(c or '').strip() for c in row)
                    for row in table if any(c and str(c).strip() for c in row)
                ]
                if len(rows) > 1:
                    tables.append('[ТАБЛИЦА]\n' + '\n'.join(rows))
    except Exception as e:
        # Логирование ошибки можно добавить позже
        pass
    return tables

# -----------------------------------------------------------
#  Функции для определения CLI-команд
# -----------------------------------------------------------
def is_cli_cmd(text: str, mono: bool) -> bool:
    """Проверяет, является ли строка началом CLI-команды."""
    t = text.strip().lower()
    return (mono or any(t.startswith(v) for v in CLI_VERBS)) and len(text) < 180

print("✅ Вспомогательные функции загружены")

✅ Вспомогательные функции загружены


In [6]:
# @title 4. Парсер документа "01 Подготовка к работе" (основная функция)

def parse_doc_01(start_id: int = 0) -> Tuple[List[Chunk], int]:
    """
    Извлекает текст, таблицы, CLI-команды из PDF-документа,
    разбивает на чанки и сохраняет иерархию заголовков.

    Args:
        start_id: начальный идентификатор для чанков (для объединения с другими документами).

    Returns:
        chunks: список объектов Chunk.
        next_id: следующий свободный идентификатор.
    """
    pdf_path = DOC_01_PATH
    doc_name = '01_Подготовка_к_работе'
    chunks   = []
    cid      = start_id
    section  = ''
    subsection = ''

    doc = fitz.open(pdf_path)
    page_h = doc[0].rect.height
    avg_sz = get_avg_font_size(doc)
    print(f'  avg_font={avg_sz:.1f}pt  pages={len(doc)}  footer_threshold_y={page_h - FOOTER_ZONE:.0f}')

    # --- Аккумулятор для накопления текста между блоками ---
    acc_text = ''
    acc_type = 'text'
    acc_page = 1
    acc_cmd  = ''
    prev_cli = False

    def flush():
        """Сброс накопленного текста в чанк(и)."""
        nonlocal cid, acc_text, acc_type, acc_cmd, prev_cli
        text = acc_text.strip()
        if not text or len(text) < MIN_TEXT:
            acc_text, acc_cmd, prev_cli = '', '', False
            return
        hdr = make_header(doc_name, acc_page, section, subsection,
                          cmd=acc_cmd if acc_type == 'cli' else '')
        parts = [text] if acc_type in ('cli', 'table', 'heading') else split_text(text)
        for part in parts:
            if len(part) >= MIN_TEXT:
                chunks.append(Chunk(
                    chunk_id=cid, text=part, text_ctx=hdr + part,
                    doc_name=doc_name, page=acc_page,
                    section=section, subsection=subsection,
                    cmd=acc_cmd, chunk_type=acc_type,
                ))
                cid += 1
        acc_text, acc_cmd, prev_cli = '', '', False

    # --- Основной цикл по страницам ---
    for page_idx in range(len(doc)):
        page = doc[page_idx]
        page_dict = page.get_text('dict', flags=fitz.TEXT_PRESERVE_LIGATURES)

        # --- Пропуск страниц оглавления ---
        page_lines = []
        for block in page_dict['blocks']:
            if block['type'] != 0:
                continue
            for line in block['lines']:
                text_line = ' '.join(span['text'] for span in line['spans']).strip()
                if text_line:
                    page_lines.append(text_line)
        # если много строк заканчиваются числом → оглавление
        toc_like = sum(1 for t in page_lines if re.search(r'\s\d{1,4}$', t))
        if toc_like > 10:
            continue

        # --- Обработка блоков страницы ---
        for block in page_dict['blocks']:
            if block['type'] != 0:
                continue

            # отсекаем колонтитулы
            y0_b, y1_b = block['bbox'][1], block['bbox'][3]
            if y1_b <= HEADER_ZONE or y0_b >= page_h - FOOTER_ZONE:
                continue

            # собираем текст и характеристики блока
            parts_text = []
            fsize = avg_sz
            bold  = False
            mono  = False
            font  = ''
            for line in block['lines']:
                for span in line['spans']:
                    t = span['text'].strip()
                    if t:
                        parts_text.append(t)
                        fsize = span['size']
                        bold  = bold or bool(span['flags'] & 16)
                        font  = span.get('font', '')
                        mono  = mono or font_is_mono(font)

            full = ' '.join(parts_text).strip()
            if not full or RE_NOISE.match(full):
                continue
            if full.strip().lower() == "содержание":
                continue
            if fsize < SZ_BODY_MIN:
                continue

            # классификация
            hlevel  = classify_heading(full, fsize, font, bold)
            is_cmd  = is_cli_cmd(full, mono)
            is_cont = bool(RE_CLI_CONT.match(full))

            # --- Логика аккумулятора ---
            if hlevel:
                flush()
                # обновление иерархии разделов
                if hlevel == 'h1':
                    section, subsection = full, ''
                elif hlevel == 'h2':
                    if not section:
                        section = full
                    else:
                        subsection = full
                else:  # h3
                    subsection = full

                hdr = make_header(doc_name, page_idx+1, section, subsection)
                chunks.append(Chunk(
                    chunk_id=cid, text=full, text_ctx=hdr + full,
                    doc_name=doc_name, page=page_idx+1,
                    section=section, subsection=subsection,
                    cmd='', chunk_type='heading',
                ))
                cid += 1
                acc_page = page_idx + 1
                prev_cli = False

            elif is_cmd and not prev_cli:
                flush()
                acc_text = full
                acc_type = 'cli'
                acc_page = page_idx + 1
                acc_cmd  = re.split(r'[<[─{\s]', full)[0].strip()
                prev_cli = True

            elif prev_cli and (is_cont or is_cmd or mono):
                acc_text += '\n' + full
                prev_cli  = True

            else:
                if prev_cli:
                    flush()
                if not acc_text:
                    acc_page = page_idx + 1
                    acc_type = 'text'
                acc_text += (' ' if acc_text else '') + full
                prev_cli  = False
                if len(acc_text) > CHUNK_SIZE * 4:
                    flush()

        del page_dict
        page = None

        # --- Обработка таблиц на странице ---
        for tbl_text in get_tables_on_page(pdf_path, page_idx):
            flush()
            hdr = make_header(doc_name, page_idx+1, section, subsection)
            chunks.append(Chunk(
                chunk_id=cid, text=tbl_text, text_ctx=hdr + tbl_text,
                doc_name=doc_name, page=page_idx+1,
                section=section, subsection=subsection,
                cmd='', chunk_type='table',
            ))
            cid += 1

        # периодическая сборка мусора
        if page_idx % 20 == 0:
            gc.collect()

    # финальный сброс
    flush()
    doc.close()
    del doc
    gc.collect()
    return chunks, cid

print("✅ Парсер определён")

✅ Парсер определён


In [ ]:
# @title (Необязательно) Переустановка PyMuPDF на случай конфликтов
# В обычных условиях не требуется, так как pymupdf уже установлен в ячейке 1.
# Раскомментируйте, если возникли проблемы с импортом fitz.

# !pip uninstall -y fitz PyMuPDF
# !pip install PyMuPDF
# import fitz

Found existing installation: PyMuPDF 1.27.1
Uninstalling PyMuPDF-1.27.1:
  Successfully uninstalled PyMuPDF-1.27.1
  Using cached pymupdf-1.27.1-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (3.4 kB)
Using cached pymupdf-1.27.1-cp310-abi3-manylinux_2_28_x86_64.whl (24.9 MB)


In [50]:
# @title 5. Загрузка чанков из JSON (для использования в поиске)

import json
import os

INDEX_DIR = "/content/drive/MyDrive/VipNet_infotex/rag_index"
CHUNKS_JSON_PATH = os.path.join(INDEX_DIR, "doc_01_chunks.json")

with open(CHUNKS_JSON_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Загружено чанков из JSON: {len(chunks)}")

# Для удобства можно сохранить количество
print("Пример первого чанка:")
print(f"  page: {chunks[0]['page']}")
print(f"  text_ctx: {chunks[0]['text_ctx'][:100]}...")

✅ Загружено чанков из JSON: 422
Пример первого чанка:
  page: 1
  text_ctx: Документ: 01_Подготовка_к_работе | Страница: 1 | Команда: ViPNet
ViPNet Coordinator HW 5...


In [51]:
# @title 6. Анализ полученных чанков (статистика, заголовки, проверка мусора)

from collections import Counter

# Статистика по типам
type_stats = Counter(c.chunk_type for c in chunks_01)
print("Статистика по типам чанков:")
for typ, cnt in type_stats.items():
    print(f"  {typ}: {cnt}")

# Заголовки (если нужно)
print("\n Заголовки (heading-чанки):")
for c in chunks_01:
    if c.chunk_type == 'heading':
        level = 'H1' if c.text == c.section else 'H2' if c.text == c.subsection else 'H?'
        print(f'  стр.{c.page:3d}  {level}  {c.text[:70]}')

# Проверка на мусор
print("\n Потенциальный мусор среди заголовков:")
for c in chunks_01:
    if c.chunk_type == 'heading':
        suspicious = (
            re.match(r'^\d', c.text) or
            len(c.text) < 4 or
            '|' in c.text
        )
        if suspicious:
            print(f'  стр.{c.page:3d}  [{c.text[:70]}]')

Статистика по типам чанков:
  cli: 104
  heading: 73
  text: 222
  table: 23

 Заголовки (heading-чанки):
  стр.  1  H1  Подготовка к работе
  стр.  6  H1  Введение
  стр.  7  H1  О документе
  стр.  8  H1  Соглашения документа
  стр. 10  H1  Что нового в версии 5.3.2
  стр. 11  H1  Совместимость с продуктами ViPNet
  стр. 12  H1  Обратная связь
  стр. 13  H1  Общая информация
  стр. 14  H1  Назначение
  стр. 15  H1  Межсетевой экран
  стр. 15  H2  Прокси-сервер
  стр. 17  H1  Средство предотвращения вторжений (IPS)
  стр. 18  H1  VPN-шлюз
  стр. 18  H2  Туннелирование на сетевом уровне (L3)
  стр. 19  H2  Туннелирование на канальном уровне (L2)
  стр. 20  H2  Маршрутизатор VPN-пакетов
  стр. 20  H2  Сервер IP-адресов
  стр. 21  H2  Сервер соединений
  стр. 22  H2  Защищенный интернет-шлюз
  стр. 23  H1  Дополнительные сетевые функции
  стр. 24  H1  Кластер высокой доступности
  стр. 25  H1  Служебные функции
  стр. 25  H2  Транспортный клиент UT
  стр. 25  H2  Транспортный сервер MFTP

In [52]:
# @title 6. Анализ полученных чанков

from collections import Counter

# Статистика по типам
type_stats = Counter(c.chunk_type for c in chunks_01)
print("📊 Статистика по типам чанков:")
for typ, cnt in type_stats.items():
    print(f"  {typ}: {cnt}")

# Заголовки (если нужно)
print("\n📌 Заголовки (heading-чанки):")
for c in chunks_01:
    if c.chunk_type == 'heading':
        level = 'H1' if c.text == c.section else 'H2' if c.text == c.subsection else 'H?'
        print(f'  стр.{c.page:3d}  {level}  {c.text[:70]}')

# Проверка на мусор
print("\n⚠️ Потенциальный мусор среди заголовков:")
for c in chunks_01:
    if c.chunk_type == 'heading':
        suspicious = (
            re.match(r'^\d', c.text) or
            len(c.text) < 4 or
            '|' in c.text
        )
        if suspicious:
            print(f'  стр.{c.page:3d}  [{c.text[:70]}]')

📊 Статистика по типам чанков:
  cli: 104
  heading: 73
  text: 222
  table: 23

📌 Заголовки (heading-чанки):
  стр.  1  H1  Подготовка к работе
  стр.  6  H1  Введение
  стр.  7  H1  О документе
  стр.  8  H1  Соглашения документа
  стр. 10  H1  Что нового в версии 5.3.2
  стр. 11  H1  Совместимость с продуктами ViPNet
  стр. 12  H1  Обратная связь
  стр. 13  H1  Общая информация
  стр. 14  H1  Назначение
  стр. 15  H1  Межсетевой экран
  стр. 15  H2  Прокси-сервер
  стр. 17  H1  Средство предотвращения вторжений (IPS)
  стр. 18  H1  VPN-шлюз
  стр. 18  H2  Туннелирование на сетевом уровне (L3)
  стр. 19  H2  Туннелирование на канальном уровне (L2)
  стр. 20  H2  Маршрутизатор VPN-пакетов
  стр. 20  H2  Сервер IP-адресов
  стр. 21  H2  Сервер соединений
  стр. 22  H2  Защищенный интернет-шлюз
  стр. 23  H1  Дополнительные сетевые функции
  стр. 24  H1  Кластер высокой доступности
  стр. 25  H1  Служебные функции
  стр. 25  H2  Транспортный клиент UT
  стр. 25  H2  Транспортный сервер M

In [11]:
# @title 7. Функция просмотра чанков

def show_chunks(chunks, chunk_type=None, page=None, keyword=None, n=10):
    """Отображает указанные чанки в удобном формате."""
    filtered = chunks
    if chunk_type:
        filtered = [c for c in filtered if c.chunk_type == chunk_type]
    if page:
        filtered = [c for c in filtered if c.page == page]
    if keyword:
        filtered = [c for c in filtered if keyword.lower() in c.text.lower()]
    print(f'Найдено: {len(filtered)} (показываю {min(n, len(filtered))})\n')
    for c in filtered[:n]:
        print(f'┌─ #{c.chunk_id} | {c.chunk_type} | стр.{c.page}')
        print(f'│  section="{c.section[:45]}"')
        if c.subsection:
            print(f'│  subsection="{c.subsection[:45]}"')
        if c.cmd:
            print(f'│  cmd="{c.cmd}"')
        print(f'├{"─"*52}')
        for line in c.text[:400].split('\n'):
            print(f'│  {line}')
        print(f'└{"─"*52}\n')

# Примеры (закомментированы)
# show_chunks(chunks_01, chunk_type='heading')
# show_chunks(chunks_01, chunk_type='cli', n=3)
# show_chunks(chunks_01, page=27)
# show_chunks(chunks_01, keyword='межсетевой')

Найдено: 73 (показываю 10)

┌─ #1 | heading | стр.1
│  section="Подготовка к работе"
├────────────────────────────────────────────────────
│  Подготовка к работе
└────────────────────────────────────────────────────

┌─ #5 | heading | стр.6
│  section="Введение"
├────────────────────────────────────────────────────
│  Введение
└────────────────────────────────────────────────────

┌─ #7 | heading | стр.7
│  section="О документе"
├────────────────────────────────────────────────────
│  О документе
└────────────────────────────────────────────────────

┌─ #10 | heading | стр.8
│  section="Соглашения документа"
├────────────────────────────────────────────────────
│  Соглашения документа
└────────────────────────────────────────────────────

┌─ #22 | heading | стр.10
│  section="Что нового в версии 5.3.2"
├────────────────────────────────────────────────────
│  Что нового в версии 5.3.2
└────────────────────────────────────────────────────

┌─ #24 | heading | стр.11
│  section="Совместимо

In [12]:
with open(os.path.join(INDEX_DIR, "doc_01_chunks.pkl"), "wb") as f:
    pickle.dump(chunks_01, f)

print("Сохранено ✓")

Сохранено ✓


In [13]:
json_path = os.path.join(INDEX_DIR, "doc_01_chunks.json")

json_data = [
    {
        "chunk_id": c.chunk_id,
        "chunk_type": c.chunk_type,
        "page": c.page,
        "section": c.section,
        "subsection": c.subsection,
        "cmd": c.cmd,
        "text": c.text,
        "text_ctx": c.text_ctx
    }
    for c in chunks_01
]

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

print("JSON сохранён ✓")
print("Файл:", json_path)

JSON сохранён ✓
Файл: /content/drive/MyDrive/VipNet_infotex/rag_index/doc_01_chunks.json


In [53]:
# @title 8. Создание эмбеддингов и FAISS-индекса (на основе JSON-чанков)

# === НАСТРОЙКИ ===
MODEL_NAME = "intfloat/multilingual-e5-base"
# Используем новые имена файлов, чтобы не затереть старые (для chunks_01)
EMBEDDINGS_PATH = os.path.join(INDEX_DIR, f"doc_01_embeddings_json_{MODEL_NAME.split('/')[-1]}.npy")
FAISS_INDEX_PATH = os.path.join(INDEX_DIR, f"doc_01_faiss_json_{MODEL_NAME.split('/')[-1]}.index")

# Загружаем модель
embed_model = SentenceTransformer(MODEL_NAME)

if os.path.exists(EMBEDDINGS_PATH) and os.path.exists(FAISS_INDEX_PATH):
    print(f"🔄 Загрузка эмбеддингов и FAISS для JSON-чанков...")
    embeddings = np.load(EMBEDDINGS_PATH)
    index = faiss.read_index(FAISS_INDEX_PATH)
else:
    print(f"🔄 Создание эмбеддингов для JSON-чанков...")
    # Для E5 нужно добавлять префикс "passage: " к текстам
    texts = [f"passage: {c['text_ctx']}" for c in chunks]
    embeddings = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Эмбеддинги сохранены: {EMBEDDINGS_PATH}")

    print("🔄 Построение FAISS-индекса...")
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"FAISS-индекс сохранён: {FAISS_INDEX_PATH}")

print(f"Размер эмбеддингов: {embeddings.shape}")
print(f"Количество векторов в индексе: {index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Создание эмбеддингов для JSON-чанков...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

💾 Эмбеддинги сохранены: /content/drive/MyDrive/VipNet_infotex/rag_index/doc_01_embeddings_json_multilingual-e5-base.npy
🔄 Построение FAISS-индекса...
💾 FAISS-индекс сохранён: /content/drive/MyDrive/VipNet_infotex/rag_index/doc_01_faiss_json_multilingual-e5-base.index
Размер эмбеддингов: (422, 768)
Количество векторов в индексе: 422


In [54]:
# @title 9. Функции поиска (для словарей)

# ----- 9.1 Токенизация для BM25 -----
def tokenize(text: str):
    """Токенизация: нижний регистр, только буквы и цифры."""
    text = text.lower()
    text = re.sub(r"[^\w\d]+", " ", text)
    return text.split()

# ----- 9.2 Построение BM25 индекса (ВАЖНО на text_ctx) -----
print("🔄 Построение BM25 индекса...")
tokenized_corpus = [tokenize(c["text_ctx"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 индекс готов.")

# ----- 9.3 Dense search (FAISS) -----
def dense_search(query: str, k: int = 5):
    q_emb = embed_model.encode([f"query: {query}"], normalize_embeddings=True)
    scores, ids = index.search(q_emb, k)
    return [(scores[0][i], chunks[ids[0][i]]) for i in range(k)]

# ----- 9.4 BM25 search -----
def bm25_search(query: str, k: int = 20):
    scores = bm25.get_scores(tokenize(query))
    top_ids = np.argsort(scores)[::-1][:k]
    return [(scores[i], chunks[i]) for i in top_ids]

# ----- 9.5 Гибридный поиск с взвешенной суммой (как в лучшем результате) -----
def hybrid_weighted_search(query: str, k: int = 5, alpha: float = 0.6):
    """
    Гибридный поиск: alpha * dense + (1-alpha) * BM25 (BM25 нормализован).
    Возвращает список словарей с ключами 'score' и 'chunk'.
    """
    # Dense
    q_emb = embed_model.encode([f"query: {query}"], normalize_embeddings=True)
    dense_scores = (embeddings @ q_emb.T).squeeze()

    # BM25
    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    # Нормализуем BM25 в [0, 1]
    bm25_scores = bm25_scores / bm25_scores.max()

    # Взвешенная сумма
    hybrid_scores = alpha * dense_scores + (1 - alpha) * bm25_scores

    # Сортируем и берём top-k
    top_idx = hybrid_scores.argsort()[::-1][:k]

    return [
        {"score": float(hybrid_scores[i]), "chunk": chunks[i]}
        for i in top_idx
    ]

# ----- 9.6 (Опционально) Переранжирование CrossEncoder -----
# Можно загрузить, если нужно, но в основном пайплайне можно обойтись без него
# для скорости. Оставим для возможности экспериментов.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list, top_k: int = 3):
    """candidates — список словарей с ключом 'chunk' (как от hybrid_weighted_search)"""
    pairs = [(query, c["chunk"]["text"]) for c in candidates]
    scores = reranker.predict(pairs)
    # Добавляем оценку в каждый кандидат
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    ranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return ranked[:top_k]

print("✅ Все функции поиска загружены.")

🔄 Построение BM25 индекса...
✅ BM25 индекс готов.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Все функции поиска загружены.


In [55]:
# @title 10. Демонстрация работы гибридного поиска

query = "Как выключить IPS?"
results = hybrid_weighted_search(query, k=3)

print(f"🔍 Запрос: {query}\n")
for i, r in enumerate(results, 1):
    chunk = r["chunk"]
    print(f"{i}. [стр.{chunk['page']}] оценка={r['score']:.3f}")
    print(f"   {chunk['text'][:150]}...\n")

🔍 Запрос: Как выключить IPS?

1. [стр.17] оценка=0.878
   Средство предотвращения вторжений (IPS)...

2. [стр.17] оценка=0.869
   База правил IPS регулярно обновляется специалистами ИнфоТеКС. При обнаружении характерных признаков вторжения (срабатывании правила IPS) IP-пакет може...

3. [стр.50] оценка=0.837
    Обновление DPI: обновление подсистемы DPI для идентификации новых, ранее не определяемых прикладных протоколов, приложений и их групп.  IPS: обрабо...



In [56]:
# @title 11. Проверка метрик на бенчмарке (для верификации)

import json

with open("/content/drive/MyDrive/VipNet_infotex/benchmark_20.json", "r", encoding="utf-8") as f:
    benchmark = json.load(f)

def evaluate_hybrid(benchmark, top_k=5):
    hits, rr = 0, 0
    for item in benchmark:
        results = hybrid_weighted_search(item["question"], k=top_k)
        for i, r in enumerate(results):
            if r["chunk"]["page"] == item["page"]:
                hits += 1
                rr += 1 / (i + 1)
                break
    return {"Hit@K": hits/len(benchmark), "MRR": rr/len(benchmark)}

print("📊 Результаты на бенчмарке:")
print(evaluate_hybrid(benchmark, top_k=5))

📊 Результаты на бенчмарке:
{'Hit@K': 0.95, 'MRR': 0.7666666666666667}


In [58]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# @title 12. Saiga LLM для генерации ответов (RAG)
# ⚠️ Для запуска требуется GPU и библиотека bitsandbytes.
# В текущей сессии GPU недоступен, поэтому код отключён.

LOAD_LLM = False    # Установите True, если есть GPU и хотите запустить

if LOAD_LLM:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    # -----------------------------------------------------------
    # 1. Загрузка модели Saiga (через transformers с 8-bit)
    # -----------------------------------------------------------
    use_8bit = True
    if use_8bit:
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    else:
        quantization_config = None

    model_name = "IlyaGusev/saiga_llama3_8b"

    print("🔄 Загрузка токенизатора...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print("🔄 Загрузка модели (это может занять пару минут)...")
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16 if not use_8bit else None,
        trust_remote_code=True
    )
    llm_model.eval()
    print("✅ Модель загружена.")

    # -----------------------------------------------------------
    # 2. Функции для формирования промпта и генерации
    # -----------------------------------------------------------
    def build_saiga_rag_prompt(question, chunks, system_prompt=None):
        if system_prompt is None:
            system_prompt = (
                "Ты — Сайга, русскоязычный ассистент. Отвечай на вопрос, "
                "используя ТОЛЬКО информацию из предоставленного контекста. "
                "Если ответа нет в контексте, скажи 'В документации нет информации'."
            )
        context_parts = []
        for chunk in chunks:
            page = chunk.get('page', '?')
            text = chunk['text']
            context_parts.append(f"[Страница {page}]: {text}")
        context = "\n\n".join(context_parts)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"}
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return prompt

    def generate_saiga_answer(question, retrieved_chunks, max_new_tokens=256):
        prompt = build_saiga_rag_prompt(question, retrieved_chunks)
        inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)
        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        answer = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return answer

    # -----------------------------------------------------------
    # 3. Финальный RAG-пайплайн
    # -----------------------------------------------------------
    def rag_pipeline(question, top_k=5, max_new_tokens=256, verbose=True):
        if verbose:
            print(f" Запрос: {question}")
            print("1. Гибридный поиск...")
        search_results = hybrid_weighted_search(question, k=top_k)
        chunks_for_llm = [r["chunk"] for r in search_results]
        if verbose:
            print(f"   Найдено чанков: {len(chunks_for_llm)}")
        print("2. Формирование промпта...")
        prompt = build_saiga_rag_prompt(question, chunks_for_llm)
        if verbose:
            prompt_tokens = len(tokenizer.encode(prompt))
            print(f"   Длина промпта: {prompt_tokens} токенов")
        print("3. Генерация ответа...")
        answer = generate_saiga_answer(question, chunks_for_llm, max_new_tokens=max_new_tokens)
        if verbose:
            print("4. Готово.")
        return answer

    # -----------------------------------------------------------
    # 4. Пример использования
    # -----------------------------------------------------------
    question = "Как выключить IPS?"
    answer = rag_pipeline(question, verbose=True)
    print(f"\n Ответ модели:\n{answer}")

else:
    print("🟡 LLM не загружена. Чтобы запустить генерацию, установите LOAD_LLM = True и убедитесь, что GPU доступен.")